# 第 55 课｜回测调试、不变量与未来函数审计

本 Notebook 对应「365 天量化金融」第 55 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为可运行、可验证的量化工程教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/55_youtube-6y6MEAgElz0/index.html`
- 环境：`pkm-quant`
- 数据：离线模拟价格与故意错误的信号
- 状态：可运行学习稿；不代表生产交易系统

## 本节量化问题

异常漂亮的回测先当作错误线索。调试链条是：数据 → 特征 → 信号 → 持仓 → 成交 → 成本 → 净值，并用日志、断言和截断不变性测试定位问题。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# logging 使用结构化字段记录关键状态；本课不进入交互式断点。
import logging
logger = logging.getLogger("lesson55")
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())

## 1. shift 方向错误会直接读取未来收益

错误特征 `return.shift(-1)` 把明日收益放到今日。正确动量特征使用已经发生的过去收益；仓位再向后移动一日。

In [3]:
# 并排展示未来泄漏信号与可执行信号。
daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
prices = daily.query("symbol == 'TECH_A'").sort_values("date").set_index("date")["adj_close"]
asset_return = prices.pct_change(fill_method=None)
wrong_signal = (asset_return.shift(-1) > 0).astype(int)
past_signal = (asset_return.rolling(5, min_periods=5).mean() > 0).astype(int)
correct_position = past_signal.shift(1).fillna(0)
audit = pd.DataFrame({
    "return": asset_return,
    "wrong_future_signal": wrong_signal,
    "past_signal": past_signal,
    "correct_position": correct_position,
}).dropna().head(8)
display(audit)
assert audit["correct_position"].iloc[1] == audit["past_signal"].iloc[0]

,return,wrong_future_signal,past_signal,correct_position
date,,,,
2023-01-04,0.010518,0,0,0.0
2023-01-05,-0.015522,0,0,0.0
2023-01-06,-0.001207,0,0,0.0
2023-01-09,-0.007248,1,0,0.0
2023-01-10,0.004622,0,0,0.0
2023-01-11,-0.025753,1,0,0.0
2023-01-12,0.003132,0,0,0.0
2023-01-13,-0.003159,0,0,0.0


## 2. 截断不变性测试

若信号在 t 日只依赖 t 及以前数据，那么删除 t 之后的数据，不应改变截至 t 的信号。下面对正确函数和错误函数分别验证。

In [4]:
# 比较完整序列与截断序列在共同历史上的输出。
def causal_signal(series):
    return (series.pct_change(fill_method=None).rolling(5, min_periods=5).mean() > 0).astype(int)

def leaky_signal(series):
    # 直接返回向前移动的未来收益，截断边界必然从“已知数值”变成 NaN。
    return series.pct_change(fill_method=None).shift(-1)

cut = 300
causal_full = causal_signal(prices).iloc[:cut]
causal_truncated = causal_signal(prices.iloc[:cut])
leaky_full = leaky_signal(prices).iloc[:cut]
leaky_truncated = leaky_signal(prices.iloc[:cut])

pd.testing.assert_series_equal(causal_full, causal_truncated)
leak_detected = not leaky_full.equals(leaky_truncated)
print("正确信号截断一致:", True)
print("错误信号检测到未来依赖:", leak_detected)
assert leak_detected

正确信号截断一致: True
错误信号检测到未来依赖: True


## 3. 会计恒等式和成本单调性是不变量

组合权益应等于现金加持仓市值；在其他条件相同时，非负交易成本不应提高最终净值。

In [5]:
# 用小型交易账本验证权益恒等式，并验证加成本后净值不更高。
ledger = pd.DataFrame({
    "cash": [1000.0, 500.0, 520.0],
    "position": [0, 5, 5],
    "price": [100.0, 100.0, 102.0],
    "equity": [1000.0, 1000.0, 1030.0],
})
ledger["calculated_equity"] = ledger["cash"] + ledger["position"] * ledger["price"]
np.testing.assert_allclose(ledger["equity"], ledger["calculated_equity"])

position = correct_position.reindex(asset_return.index).fillna(0)
gross = position * asset_return.fillna(0)
turnover = position.diff().abs().fillna(position.abs())
net = gross - turnover * 0.001
gross_nav = (1 + gross).cumprod()
net_nav = (1 + net).cumprod()
assert net_nav.iloc[-1] <= gross_nav.iloc[-1] + 1e-12
logger.info("final_gross=%.4f final_net=%.4f", gross_nav.iloc[-1], net_nav.iloc[-1])
display(ledger)

final_gross=1.8703 final_net=1.6326


,cash,position,price,equity,calculated_equity
0,1000.0,0,100.0,1000.0,1000.0
1,500.0,5,100.0,1000.0,1000.0
2,520.0,5,102.0,1030.0,1030.0


## 方法边界与复用结论


- 调试器展示执行状态，但不能自动判断金融假设是否合理。
- 日志应含时间、标的、信号、委托、成交、成本和持仓，同时控制级别避免淹没信息。
- 修复错误导致收益下降是正常结果；修复后仍需独立样本外验证。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。